# EX CNN Simple - Fashion MNIST

Selles töövihikus ehitad nullist kergekaalulise konvolutsioonivõrgu, mis suudab klassifitseerida Fashion MNIST riideesemeid.
- Andmestik: `tf.keras.datasets.fashion_mnist`
- Sisendid: 28×28 halltoonpildid (1 kanal)
- Väljund: 10 klassi softmax aktivatsiooniga

Tulemuse mõistlik orientiir on >85% täpsust testandmetel, kui mudel on õigesti treenitud ja regulariseeritud.


**Soovitused:**
- Kasuta TensorFlow/Keras `Sequential` API-t koos konvolutsioonikihtide ja avatud tavaliste Dense kihtidega.
- Hoia funktsioonid malli defineeritud signatuurides, et testid saaksid neid importida.
- Kontrolli järjepidevalt massiivide kuju (`shape`) ja andmetüüpi (`dtype`), et vältida varjatud vigu.
- Kui GPU pole saadaval või soovid jooksutada teste CPU-l, keela see `CUDA_VISIBLE_DEVICES = '-1'` kaudu.
- Treeni esialgu lühema epochide arvuga või väiksema andmehulgaga, et veenduda toru toimimises.


## 0) Eeltöö: teegid ja seadistused

Enne funktsioonide täitmist koonda ühte plokki kõik vajalikud impordid (NumPy, TensorFlow, Keras kihid/utiliidid) ja globaalsed konstandid.

**Sinu ülesanne:**
- Impordi `numpy as np`, `tensorflow as tf`, `Sequential`, konvolutsioonikihtide klassid ning `to_categorical`.
- Defineeri konstandid nagu `IMAGE_SHAPE = (28, 28, 1)`, `NUM_CLASSES = 10`, `BATCH_SIZE` ja `RANDOM_SEED`.
- Sea `np.random.seed` ja `tf.random.set_seed`, et tulemused oleksid taasesitatavad.
- Soovi korral keela GPU kasutamine, et vältida erinevusi arvutites.

**Kontrolli:**
- Konstandid on hilisemates funktsioonides nähtavad.
- `IMAGE_SHAPE` kirjeldab täpselt Fashion MNIST ühe pildi mõõte.


In [1]:
"""Simplified CNN exercise template using Fashion MNIST."""

import os

import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Dropout, Flatten, MaxPooling2D
from tensorflow.keras.utils import to_categorical

# Optional CPU-only execution for reproducibility across machines.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

IMAGE_SHAPE = (28, 28, 1)
NUM_CLASSES = 10
BATCH_SIZE = 128
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


I0000 00:00:1778079368.894597   90284 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778079368.895370   90284 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778079368.937998   90284 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778079370.130600   90284 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

## Google Drive (valikuline)

Funktsioon `try_to_mount_drive` on ette antud. Seda saad kasutada, kui töötad Google Colabis ja soovid mudeleid või vahefaile Drive'i salvestada.


In [ ]:
def try_to_mount_drive():
    """Mount Google Drive. For local usage only."""
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        import os
        os.chdir('/content/drive/My Drive/Colab Notebooks')
        return True
    except ImportError:
        print("Google Colab module not found. Skipping Google Drive mount.")
        return False


## 1) Andmete laadimine ja eeltöö

Fashion MNIST sisaldab 60 000 treening- ja 10 000 testpilti. Mudel eeldab, et pildid on CNN-ile sobivas formaadis.

**Sinu ülesanne:**
- Kasuta `tf.keras.datasets.fashion_mnist.load_data()` andmete lugemiseks.
- Muuda pildid kuju `(N, 28, 28, 1)` peale ja arvuta need `float32` kujule.
- Normaliseeri pikslid vahemikku `[0, 1]` jagades väärtused 255-ga.
- Teisenda treening- ja testsildid one-hot kujule `to_categorical` abil 10 klassiga.
- Tagasta `((X_train, y_train), (X_test, y_test))`.

**Kontrolli:**
- `X_train.shape == (60000, 28, 28, 1)` ja `X_test.shape == (10000, 28, 28, 1)`.
- `np.max(X_train)` ja `np.max(X_test)` jäävad <= 1.0; `dtype` on `float32`.
- `y_train` ja `y_test` igal real on täpselt üks `1.0` ja ülejäänud `0.0` (one-hot).


In [ ]:
def load_and_process_data():
    """Load Fashion MNIST and return train/test splits ready for CNNs."""
    (X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

    X_train = X_train.reshape((-1, *IMAGE_SHAPE)).astype(np.float32) / 255.0
    X_test = X_test.reshape((-1, *IMAGE_SHAPE)).astype(np.float32) / 255.0

    y_train = to_categorical(y_train, NUM_CLASSES)
    y_test = to_categorical(y_test, NUM_CLASSES)

    return (X_train, y_train), (X_test, y_test)


## 2) CNN arhitektuur

Vaja on ühe Sequential-mudeliga lahendada klassifikatsioon.

**Sinu ülesanne:**
- Ehita mudel parameetrite `input_shape` ja `num_classes` põhjal.
- Lisa vähemalt üks `Conv2D` + `MaxPooling2D` blokk, mille järel tuleb `Flatten` ja Dense kihid.
- Kasuta `Dropout`-i või teisi reguliseerimisvõtteid, et üleõppimist vähendada.
- Lõppkiht peab olema `Dense(num_classes, activation='softmax')`.

**Kontrolli:**
- Mudelil on `predict` meetod ja piisavalt kihte (>=5) sisendi teisendamiseks.
- `model.layers` sisaldab nii konvolutsiooni kui ka basseini kihti.
- Viimase kihi üksuste arv võrdub `num_classes` väärtusega.


In [ ]:
def create_model(input_shape, num_classes):
    """Create a Sequential CNN model."""
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax'),
    ])
    return model


## 3) Mudeli kompileerimine

Enne treenimist on vaja määrata kaofunktsioon, optimeerija ja mõõdikud.

**Sinu ülesanne:**
- Kasuta `model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])` või samaväärset konfiguratsiooni.
- Tagasta sama mudeli instants, et toru saaks seda edasi kasutada.

**Kontrolli:**
- `model.loss` sisaldab fraasi `categorical_crossentropy`.
- `history` väljundis kuvatakse `accuracy` väärtused.


In [ ]:
def compile_model(model):
    """Compile the CNN model with an optimizer, loss, and metrics."""
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


## 4) Mudeli treenimine

`train_model` peaks kapseldama kõik `model.fit` detailid.

**Sinu ülesanne:**
- Konverteeri sisendid NumPy massiivideks (`dtype=np.float32`).
- Käivita `model.fit` koos `validation_data=(X_test, y_test)` ja parameetritega `epochs` ning `batch_size`.
- Tagasta Keras `History` objekt, et hiljem saaks tulemusi visualiseerida.

**Kontrolli:**
- `history.history` sisaldab võtmeid `loss`, `val_loss`, `accuracy` jne.
- Treening ei viska erindit ja kasutab mõistlikku partiisuurust.


In [ ]:
def train_model(model, X_train, X_test, y_train, y_test, epochs=20, batch_size=128):
    """Train the CNN using the provided train/test split."""
    X_train = np.asarray(X_train, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.float32)

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_test, y_test),
        epochs=epochs,
        batch_size=batch_size,
        verbose=1,
    )
    return history


## 5) Mudeli hindamine (`model.evaluate`)

Funktsioon peab andma täpsuse väärtuse, millele testid tuginevad.

**Sinu ülesanne:**
- Konverteeri sisendid `np.asarray(..., dtype=np.float32)` abil.
- Kasuta `model.evaluate` ja tagasta ainult täpsus (teine element tuple'ist).

**Kontrolli:**
- Tagastus jääb vahemikku 0...1 ja on float.
- Funktsioon ei prindi üleliigset infot.


In [ ]:
def evaluate_model(model, X_test, y_test):
    """Evaluate the model on the test split using model.evaluate."""
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.float32)
    _, accuracy = model.evaluate(X_test, y_test, verbose=0)
    return float(accuracy)


## 6) Ennustustel põhinev hindamine

Võrdle `model.predict` väljundit tegelike one-hot vektoritega, et kinnitada `model.evaluate` tulemust.

**Sinu ülesanne:**
- Tee `model.predict(X_test)` (vaikimisi `verbose=0`).
- Leia `np.argmax` abil nii ennustatud kui ka tegelikud klassid.
- Arvuta täpsus `np.mean(predicted_classes == true_classes)`.

**Kontrolli:**
- Saadud täpsus on väga lähedane `evaluate_model` tulemusest (~±0.05).


In [ ]:
def evaluate_using_predictions(model, X_test, y_test):
    """Compute accuracy by comparing predicted classes to labels."""
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.float32)

    y_pred = model.predict(X_test, verbose=0)
    predicted_classes = np.argmax(y_pred, axis=1)
    true_classes = np.argmax(y_test, axis=1)

    accuracy = np.mean(predicted_classes == true_classes)
    return float(accuracy)


## 7) Peaprogramm

Koosta terviklik toru, mis seob kõik funktsioonid kokku ning väljastab lühikese kokkuvõtte.

**Sinu ülesanne:**
- Lae andmed ja jaga need treening/valideerimise hulka.
- Leia mudeli sisendkuju ja klasside arv dünaamiliselt.
- Loo, kompileeri ja treeni mudel, seejärel kuva mõlemad täpsused.
- Vajadusel prindi mudeli kokkuvõte või treeningu pikkus, et tulemusi jälgida.

**Kontrolli:**
- `main()` töötab otsast lõpuni ilma käsitsi sekkumiseta.
- Konsooli väljastus sisaldab vähemalt kahte numbrit (evaluate + predict täpsus).


In [ ]:
def main():
    try_to_mount_drive()
    (X_train, y_train), (X_test, y_test) = load_and_process_data()
    model = create_model(X_train.shape[1:], y_train.shape[1])
    compile_model(model)
    history = train_model(model, X_train, X_test, y_train, y_test, epochs=5)
    accuracy = evaluate_model(model, X_test, y_test)
    prediction_accuracy = evaluate_using_predictions(model, X_test, y_test)

    print(f'Training epochs: {len(history.history.get("loss", []))}')
    print(f'Accuracy (evaluate): {accuracy * 100:.2f}%')
    print(f'Accuracy (predict): {prediction_accuracy * 100:.2f}%')

if __name__ == '__main__':
    main()
